# Python Generators: Fibonacci Three Ways

This notebook compares three approaches to producing Fibonacci numbers in Python:

1. **A regular function** that builds and returns a full list (eager evaluation)
2. **A recursive function** that computes numbers via function calls
3. **A generator function** (`yield`) that produces numbers lazily, one at a time

Along the way we'll compare **memory usage**, **speed**, and **when each approach makes sense** — including why generators are a classic "big data" technique, in the same spirit as `pandas`' `chunksize` or PySpark's lazy evaluation covered elsewhere.

## 1. Regular function — build the whole list upfront

The simplest approach: compute all `n` Fibonacci numbers in a loop and return them as a list. This is **eager** — every value is computed and held in memory before you get anything back.

In [1]:
def fib_list(n):
    """Return the first n Fibonacci numbers as a list (eager)."""
    result = []
    a, b = 0, 1
    for _ in range(n):
        result.append(a)
        a, b = b, a + b
    return result

# Try it
print(fib_list(10))


[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


## 2. Recursive function

A more "mathematical" way to express Fibonacci: `fib(k) = fib(k-1) + fib(k-2)`. This mirrors the definition directly, but naive recursion **recomputes the same values many times** — its time complexity is exponential, O(2^k), because `fib(k-1)` and `fib(k-2)` each independently recompute overlapping subproblems.

We'll write two versions:
- `fib_recursive(k)` — returns the k-th Fibonacci number only, naive recursion
- `fib_recursive_list(n)` — builds a list of the first n numbers by calling the naive recursive function n times (to keep the comparison apples-to-apples with `fib_list`)

In [2]:
def fib_recursive(k):
    """Return the k-th Fibonacci number using naive recursion (no memoization)."""
    if k < 2:
        return k
    return fib_recursive(k - 1) + fib_recursive(k - 2)

def fib_recursive_list(n):
    """Return the first n Fibonacci numbers using naive recursion for each one."""
    return [fib_recursive(k) for k in range(n)]

# Try it — keep n small, naive recursion gets slow fast!
print(fib_recursive_list(10))


[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


**Note:** naive recursion is elegant but impractical for anything beyond small `n` — we'll demonstrate exactly how slow it gets in the timing section below. (A memoized or iterative version fixes the speed problem, but doesn't change the *eager, all-at-once* memory profile we're focusing on here.)

## 3. Generator function — lazy, one value at a time

A generator function looks like a normal function but uses `yield` instead of `return`. Calling it doesn't run the function body at all — it returns a **generator object**, an iterator that runs the code only up to the next `yield` each time you ask it for a value with `next()` (or a `for` loop).

This means:
- No list is ever fully built in memory
- You can ask for "the next value" indefinitely, even from an **infinite** sequence
- Values are computed **on demand** — this is the same lazy-evaluation idea behind Spark's DAGs and pandas' `chunksize` iterator

In [3]:
def fib_generator(n=None):
    """Yield Fibonacci numbers one at a time.
    If n is given, stop after n numbers. If n is None, yield forever.
    """
    a, b = 0, 1
    count = 0
    while n is None or count < n:
        yield a
        a, b = b, a + b
        count += 1

# Calling it does NOT compute anything yet -- it just creates a generator object
gen = fib_generator(10)
print(gen)          # <generator object ...>
print(list(gen))    # now we actually pull the values


<generator object fib_generator at 0x7fc33919e6c0>
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


In [4]:
# Generators are pulled one value at a time with next()
gen = fib_generator(5)
print(next(gen))
print(next(gen))
print(next(gen))
# ... and so on. The function's local state (a, b, count) is remembered between calls.


0
1
1


In [5]:
# Because it can run forever, we can use it as a genuinely infinite stream,
# and only take what we need -- e.g. with itertools.islice
import itertools

infinite_fib = fib_generator()  # no limit
first_15 = list(itertools.islice(infinite_fib, 15))
print(first_15)

# the SAME generator object remembers where it left off:
next_5 = list(itertools.islice(infinite_fib, 5))
print(next_5)


[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377]
[610, 987, 1597, 2584, 4181]


## 4. A recursive *generator*

Recursion and generators aren't mutually exclusive — you can write a recursive function that yields, using `yield from` to delegate to nested generator calls. This is rarely the *fastest* way to generate Fibonacci numbers, but it's a nice illustration that "lazy" and "recursive" are independent ideas that can be combined.

In [6]:
def fib_recursive_generator(n, _cache={}):
    """Yield the first n Fibonacci numbers, computing each via memoized recursion."""
    def fib(k):
        if k < 2:
            return k
        if k not in _cache:
            _cache[k] = fib(k - 1) + fib(k - 2)
        return _cache[k]

    for k in range(n):
        yield fib(k)

print(list(fib_recursive_generator(10)))


[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


Note the `_cache` here is a form of **memoization** — without it, a recursive generator built on `fib_recursive` (no caching) would still be exponentially slow per value, even though it's "lazy" about which values it computes. Laziness (generators) and efficiency (memoization/iteration) solve two *different* problems, and it's easy to assume a generator automatically fixes an underlying algorithmic problem when it doesn't.

## 5. Memory comparison

This is the headline benefit of generators. A list holding `n` items must allocate space for all of them at once; a generator holds only its current state (a couple of integers here), regardless of how many values you eventually pull from it.

In [7]:
import sys

n = 100_000

lst = fib_list(n)
gen = fib_generator(n)

print(f"List of {n:,} Fibonacci numbers:      {sys.getsizeof(lst):>12,} bytes")
print(f"Generator object (same count):       {sys.getsizeof(gen):>12,} bytes")


List of 100,000 Fibonacci numbers:           800,984 bytes
Generator object (same count):                216 bytes


The generator's size **does not depend on `n`** — try changing `n` above to `1_000_000` or `100_000_000` and the generator's footprint stays essentially constant, while the list's footprint grows linearly (and for Fibonacci numbers, the *integers themselves* also grow huge, adding even more memory the list must hold that the generator only ever holds one of at a time).

In [8]:
# Confirming the generator's memory footprint stays flat as n grows
for n in [10, 10_000, 10_000_000]:
    g = fib_generator(n)
    print(f"n={n:>12,}  ->  generator size: {sys.getsizeof(g)} bytes")


n=          10  ->  generator size: 216 bytes
n=      10,000  ->  generator size: 216 bytes
n=  10,000,000  ->  generator size: 216 bytes


## 6. Speed comparison

Two different things are being timed here, so it's worth separating them clearly:

- **`fib_list` vs `fib_generator`**, fully consumed: both are O(n) with tiny constant-factor differences — the generator isn't "faster" at producing every value, its advantage is memory, not raw speed, when you need *all* the values anyway.
- **Naive recursion vs the iterative approaches**: this is where the real speed gap shows up — O(2^k) vs O(n).

In [9]:
import time

def time_it(label, fn):
    start = time.perf_counter()
    fn()
    elapsed = time.perf_counter() - start
    print(f"{label:<40} {elapsed:.4f} s")

n = 25_000

time_it("fib_list (eager list)",            lambda: fib_list(n))
time_it("fib_generator (fully consumed)",   lambda: list(fib_generator(n)))


fib_list (eager list)                    0.0244 s
fib_generator (fully consumed)           0.0182 s


In [10]:
# Naive recursion is fine for small n... 
n_small = 25
time_it(f"fib_recursive_list (naive recursion, n={n_small})", lambda: fib_recursive_list(n_small))

# ...but gets dramatically slower as n grows, because of repeated recomputation.
# Uncomment to see for yourself (this can take a while beyond ~n=30-32!):
# n_bigger = 32
# time_it(f"fib_recursive_list (naive recursion, n={n_bigger})", lambda: fib_recursive_list(n_bigger))


fib_recursive_list (naive recursion, n=25) 0.0125 s


## 7. When to reach for a generator

| Approach | Memory | Speed (computing all n values) | Best for |
|---|---|---|---|
| **Regular function (list)** | O(n) — all values held at once | O(n) | Small/medium `n` where you need random access or to reuse the values multiple times |
| **Naive recursion** | O(n) call stack per value (plus O(n) if collected into a list) | O(2^k) per value — exponential, impractical beyond small k | Illustrating the mathematical definition; not for production use without memoization |
| **Generator (`yield`)** | O(1) — only current state is held | O(n) | Large or unbounded sequences, streaming data, pipelines where you only need to iterate once, or where you might stop early |

The same trade-off shows up in the bigger "big data" techniques covered elsewhere: pandas' `chunksize` (a generator over file chunks) and PySpark's lazy evaluation (build a plan, execute on demand) are both instances of this same core idea — **don't materialize more data than you need to have in memory at once.**